<div align="center">
  <img src="https://raw.githubusercontent.com/UN-GCPDS/BananaRipenessCREDA/main/assets/banner.png" alt="BananaRipenessCREDA Banner" width="850"/>
</div>

# **Taller Práctico IEEE: De la Teoría al Borde (Edge AI)**
## **Clasificación y Cuantización de Madurez de Bananos con MobileNetV3, CREDA y ExecuTorch**

---

### **Universidad Estatal Península de Santa Elena (UPSE)**
#### **Facultad de Sistemas y Telecomunicaciones**
#### **Rama Estudiantil IEEE UPSE**

* **Docentes a cargo:**
  * **Prof. Luis Chuquimarca** (`lchuquimarca@upse.edu.ec`)
  * **Prof. Lucas Iturriago** (`liturriago@unal.edu.co`)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UN-GCPDS/BananaRipenessCREDA/blob/main/notebooks/Workshop/Taller_IEEE_BananaRipenessCREDA.ipynb)
![Python](https://img.shields.io/badge/Python-3.10%2B-blue?logo=python)
![PyTorch](https://img.shields.io/badge/PyTorch-2.0%2B-EE4C2C?logo=pytorch)
![ExecuTorch](https://img.shields.io/badge/ExecuTorch-INT8%20XNNPACK-orange)
![HuggingFace](https://img.shields.io/badge/HuggingFace-Spaces%20%26%20Gradio-FFD21E?logo=huggingface)

---

### **Descripción y Objetivos del Taller**

En aplicaciones agrícolas e industriales de Visión Artificial, los modelos entrenados en laboratorios o imágenes sintéticas a menudo sufren una caída drástica de rendimiento al desplegarse en entornos reales debido a variaciones de iluminación, sombras y fondos (**Domain Shift**). Además, el despliegue en dispositivos de bajo consumo (**Edge Devices** como Raspberry Pi o microcontroladores) exige optimizar drásticamente la latencia y la memoria.

En este taller aprenderás a:
1. **Configurar el entorno en Google Colab** y vincular el dataset oficial de madurez de bananos (`bananaripeness`) alojado en Kaggle Hub.
2. **Entrenar una arquitectura MobileNetV3** bajo el paradigma de **Adaptación de Dominio No Supervisada (UDA)** utilizando el algoritmo **CREDA** (*Class-Regularized Entropy Domain Adaptation*).
3. **Evaluar rigurosamente el modelo** mediante métricas científicas (Precision, Recall, F1-Score), Matrices de Confusión, Curvas ROC y visualizaciones latentes con **UMAP**.
4. **Convertir y Cuantizar el modelo a INT8 con ExecuTorch (PT2E + XNNPACK)**, analizando la reducción de memoria y la aceleración en CPU.
5. **Desplegar la solución en Hugging Face Spaces** con una interfaz interactiva en **Gradio**, dotada de arquitectura Docker y un mecanismo de fallback resiliente.
6. **Consumir la API remota del Space** programáticamente utilizando `gradio_client`.

---
# 1. Preparación del Entorno en Google Colab

Primero comprobaremos la disponibilidad de aceleración por GPU (CUDA) y clonaremos el repositorio del proyecto.

In [ ]:
# 1.1. Verificar acelerador de hardware GPU asignado por Colab
!nvidia-smi

## 1.2. Clonación e Instalación de Dependencias

Instalaremos el paquete modular `banana_creda` en modo editable junto con sus extensiones de cuantización (`convert`) y explicabilidad (`explain`), además de librerías para despliegue y descarga de datasets.

In [ ]:
import os
import sys

# Clonar el repositorio oficial si no existe localmente
if not os.path.exists("BananaRipenessCREDA"):
    print("[*] Clonando repositorio BananaRipenessCREDA...")
    !git clone https://github.com/UN-GCPDS/BananaRipenessCREDA.git
    %cd BananaRipenessCREDA
else:
    %cd BananaRipenessCREDA

print(f"[*] Directorio de trabajo actual: {os.getcwd()}")

# Instalar dependencias del proyecto
!pip install -e .[convert,explain,dev] -q
!pip install kagglehub gradio gradio_client huggingface_hub pyyaml -q

## 1.3. Descarga y Verificación del Dataset

Utilizaremos `kagglehub` para descargar el dataset `lucasiturriago/bananaripeness`.
Los datos se ubicarán en el directorio estándar de cache de Kaggle:
`/root/.cache/kagglehub/datasets/lucasiturriago/bananaripeness/versions/1`

In [ ]:
import kagglehub
from pathlib import Path

# Descargar el dataset oficial
print("[*] Descargando dataset desde Kaggle Hub...")
downloaded_path = kagglehub.dataset_download("lucasiturriago/bananaripeness")
DATASET_DIR = Path("/root/.cache/kagglehub/datasets/lucasiturriago/bananaripeness/versions/1")

# En caso de que se ejecute en un entorno donde la ruta varíe:
if not DATASET_DIR.exists() and Path(downloaded_path).exists():
    DATASET_DIR = Path(downloaded_path)

print(f"\n[+] Directorio del dataset: {DATASET_DIR}")
print("[+] Estructura de carpetas disponibles:")
for folder in sorted(DATASET_DIR.iterdir()):
    if folder.is_dir():
        samples_count = len(list(folder.rglob("*.jpg")) + list(folder.rglob("*.png")))
        print(f"  📂 {folder.name:<20} -> {samples_count} imágenes")

---
# 2. Configuración Dinámica del Experimento (MobileNetV3 + CREDA)

Para este taller utilizaremos el backbone **MobileNetV3**, diseñado específicamente para maximizar la precisión mientras mantiene una baja latencia en procesadores móviles y embebidos.

Las clases de madurez corresponden a:
* **Class A**: Fruto inmaduro / Verde (*Unripe*).
* **Class B**: Inicio de maduración / Verde-Amarillo (*Barely Ripe*).
* **Class C**: Maduro óptimo para consumo fresco (*Ripe*).
* **Class D**: Sobre-maduro / Manchas oscuras (*Overripe*).

Generaremos un archivo de configuración YAML validado por los esquemas Pydantic del paquete `banana_creda`.

In [ ]:
import yaml

# Definición declarativa de la configuración del experimento (CREDA - Medium Variation)
colab_config = {
    "data": {
        "source_data_dir": str(DATASET_DIR / "Original"),
        "target_data_dir": str(DATASET_DIR / "Medium_Variation"),
        "batch_size": 32,
        "img_size": 224,
        "num_workers": 2,
        "imagenet_mean": [0.485, 0.456, 0.406],
        "imagenet_std": [0.229, 0.224, 0.225],
        "use_lime_on_target": True,
        "use_augmentation": True
    },
    "model": {
        "num_classes": 4,
        "pretrained": True,
        "backbone": "mobilenetv3",
        "dropout_rate": 0.2
    },
    "training": {
        "epochs": 20,
        "lr": 0.0001,
        "gamma": 0.94,
        "warmup": True,
        "warmup_epochs": 5,
        "warmup_threshold": 0.9,
        "lambda_creda": 1.0,
        "use_uncertainty": True,
        "sigma": "auto",
        "use_amp": True,
        "device": "cuda",
        "seed": 42
    },
    "experiment": {
        "name": "medium_variation_mobilenetv3_experiment",
        "version": 3,
        "output_dir": "outputs/mobilenetv3/experiment_3",
        "save_results": True
    }
}

CONFIG_FILE = "configs/training/mobilenetv3/workshop_colab.yaml"
os.makedirs(os.path.dirname(CONFIG_FILE), exist_ok=True)

with open(CONFIG_FILE, "w", encoding="utf-8") as f:
    yaml.dump(colab_config, f, sort_keys=False, default_flow_style=False)

print(f"[+] Archivo de configuración generado exitosamente: {CONFIG_FILE}")

---
# 3. Entrenamiento con Adaptación de Dominio (CREDA)

Ejecutaremos el script [`scripts/da_train.py`](file:///scripts/da_train.py). Durante el proceso:
1. Se inicializa el modelo pre-entrenado `mobilenet_v3_large`.
2. Se activa el cálculo de la divergencia de Rényi condicional de orden 2 con kernel RBF adaptativo para alinear las características del dominio sintético/estudio con el dominio de iluminación variable.
3. Los mejores pesos se guardan en `outputs/mobilenetv3/experiment_3/model_final.pth`.

In [ ]:
# Ejecutar entrenamiento de Domain Adaptation (CREDA con MobileNetV3 en Medium_Variation)
!python scripts/da_train.py --config configs/training/mobilenetv3/workshop_colab.yaml

---
# 4. Evaluación Científica del Modelo

Procederemos a evaluar el modelo final sobre el conjunto de prueba del dominio destino (Target Test Set), midiendo exactitud, precisión, recall, F1-score por clase y generando las visualizaciones de diagnóstico.

In [ ]:
# 4.1. Ejecución del script de evaluación
!python scripts/evaluation.py \
    --config configs/training/mobilenetv3/workshop_colab.yaml \
    --model outputs/mobilenetv3/experiment_3/model_final.pth \
    --output_dir outputs/mobilenetv3/experiment_3

## 4.2. Visualización Diagnóstica de Resultados

### 4.2.1. Métricas Cuantitativas: Matriz de Confusión y Curva ROC

La matriz de confusión permite verificar la tasa de aciertos y posibles confusiones entre clases adyacentes (por ejemplo, entre Fruto Verde e Inicio de Maduración). La curva ROC multicategoría demuestra la capacidad discriminativa del modelo en función de la variación de umbrales de decisión.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

eval_dir = Path("outputs/mobilenetv3/experiment_3/evaluation")
if not eval_dir.exists():
    eval_dir = Path("outputs/mobilenetv3/experiment_3")

# Visualizar Matriz de Confusión
cm_plots = list(eval_dir.glob("*confusion_matrix*.png"))
if cm_plots:
    print(f"[+] Matriz de Confusión ({cm_plots[0].name}):")
    display(Image(filename=str(cm_plots[0]), width=700))

# Visualizar Curva ROC
roc_plots = list(eval_dir.glob("*roc_curve*.png"))
if roc_plots:
    print(f"[+] Curva ROC Multiclase ({roc_plots[0].name}):")
    display(Image(filename=str(roc_plots[0]), width=700))

### 4.2.2. Proyección Latente UMAP: Alineación de Dominios (CREDA)

**¿Qué representa esta gráfica?**
UMAP (*Uniform Manifold Approximation and Projection*) reduce la dimensión de las características profundas extraídas por la penúltima capa de MobileNetV3 (1280 dimensiones) a un espacio visual 2D:
* **Dominio Origen (Source / Original):** Muestras capturadas bajo condiciones homogéneas de luz.
* **Dominio Destino (Target / Medium Variation):** Muestras capturadas bajo variaciones lumínicas complejas (sombras, saturación, baja luminosidad).

**Criterio de Éxito en Adaptación de Dominio:**
Si el algoritmo **CREDA** ha convergido exitosamente, los puntos de ambos dominios se encontrarán **superpuestos armónicamente por cada estado de madurez**, demostrando que la red ha aprendido representaciones semánticas invariantes al cambio de iluminación.

In [ ]:
# 1. Gráfica UMAP de Alineación de Distribuciones (Source vs. Target)
umap_alignment_plots = list(eval_dir.glob("*umap_alignment*.png"))
if umap_alignment_plots:
    print(f"\n[+] UMAP de Alineación de Dominio: {umap_alignment_plots[0].name}")
    display(Image(filename=str(umap_alignment_plots[0]), width=750))
else:
    print("[!] No se encontró la gráfica de alineación UMAP.")

# 2. Gráfica UMAP Cualitativa con Muestras Representativas de Imágenes
umap_images_plots = list(eval_dir.glob("*umap_with_images*.png")) or list(eval_dir.glob("*Latent Space*.png"))
if umap_images_plots:
    print(f"\n[+] UMAP Cualitativo con Imágenes Proyectadas: {umap_images_plots[0].name}")
    display(Image(filename=str(umap_images_plots[0]), width=850))
else:
    print("[INFO] Ejecuta el análisis cualitativo o scripts/explain.py para generar el mapa con imágenes embebidas.")

---
# 5. Conversión y Cuantización a INT8 con ExecuTorch (PT2E + XNNPACK)

### **¿Por qué ExecuTorch y Cuantización INT8?**
* **ExecuTorch** es el runtime de nueva generación de PyTorch diseñado específicamente para dispositivos de borde (teléfonos móviles, wearables y SBCs como Raspberry Pi).
* **Cuantización INT8:** Reduce la precisión de los pesos y activaciones de 32 bits en coma flotante (FP32) a enteros de 8 bits (INT8). Esto permite:
  * **Reducción de tamaño en ~75%:** Menor consumo de memoria flash y RAM.
  * **Mayor velocidad:** Uso eficiente de instrucciones SIMD vectoriales de la CPU (ARM NEON / x86 AVX) a través del backend de alto rendimiento **XNNPACK**.

Ejecutaremos [`scripts/convert_and_quantize.py`](file:///scripts/convert_and_quantize.py), el cual realiza:
1. Exportación de grafo ATen con `torch.export`.
2. Inserción de observadores y calibración estática con 100 muestras reales.
3. Conversión a operadores cuantizados y lowering a formato binario `.pte`.
4. Comparativa cuantitativa de latencia y métricas.

In [ ]:
# Ejecutar conversión y cuantización PT2E con backend XNNPACK
!python scripts/convert_and_quantize.py \
    --config configs/training/mobilenetv3/workshop_colab.yaml \
    --model outputs/mobilenetv3/experiment_3/model_final.pth

---
# 6. Despliegue en Hugging Face Spaces con Gradio y Docker

Una vez obtenido el modelo optimizado de ExecuTorch (`model_quantized_xnnpack.pte`) y los pesos PyTorch (`model_final.pth`), crearemos un **Hugging Face Space** interactivo.

### **Arquitectura del Space:**
* **Frontend:** Gradio Blocks interactivo que permite arrastrar imágenes de bananos y devuelve la clase predicha con su respectiva interpretación comercial.
* **Backend:** Servidor Docker en Python 3.10 con el runtime compilado de ExecuTorch y backend XNNPACK.
* **Mecanismo de Resiliencia (Fallback):** Si la arquitectura del host remoto en Hugging Face no soporta el runtime binario de ExecuTorch en C++, la aplicación conmuta automáticamente al modelo PyTorch FP32 para asegurar 100% de disponibilidad sin interrupciones.

## 6.1. Autenticación en Hugging Face

Ejecuta la siguiente celda e introduce tu **User Access Token** (con permisos de *write*) obtenido en [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login

# Inicia sesión en tu cuenta de Hugging Face
notebook_login()

## 6.2. Parámetros del Space

Configura tu nombre de usuario de Hugging Face y el nombre para el Space.

In [ ]:
HF_USER = "tu_usuario_hf"  # <-- REEMPLAZA CON TU USUARIO DE HUGGING FACE
SPACE_NAME = "banana-ripeness-detector"
SPACE_DIR = Path(SPACE_NAME)
SPACE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Space configurado: https://huggingface.co/spaces/{HF_USER}/{SPACE_NAME}")

## 6.3. Generación de `app.py` (Gradio con Runtime de ExecuTorch + Fallback)

Escribimos el código de la aplicación web que gestionará la inferencia en tiempo real.

In [ ]:
app_code = """import os
import sys
import numpy as np
import torch
import gradio as gr
from PIL import Image

# -------------------------------------------------------------
# 0. Soporte para Entornos con ZeroGPU (Hugging Face Spaces)
# -------------------------------------------------------------
try:
    import spaces
    print("[INFO] Módulo 'spaces' (ZeroGPU) detectado y cargado.")
except ImportError:
    class spaces:
        @staticmethod
        def GPU(func=None, **kwargs):
            if func is not None:
                return func
            def decorator(f):
                return f
            return decorator
    print("[INFO] Ejecutando en entorno estándar CPU / No-ZeroGPU.")

# -------------------------------------------------------------
# 1. Carga de Librerías y Runtime ExecuTorch
# -------------------------------------------------------------
try:
    from executorch.runtime import Runtime
    print("[INFO] Runtime nativo de ExecuTorch cargado exitosamente.")
except ImportError:
    raise ImportError("[ERROR] No se encontró el runtime de ExecuTorch. Verifica que 'executorch' esté en requirements.txt")

from torchvision import transforms

# Ruta del modelo optimizado INT8 (Único artefacto requerido para Edge AI)
PATH_PTE = "model_quantized_xnnpack.pte"

# -------------------------------------------------------------
# 2. Definición de Clases e Información de Madurez
# -------------------------------------------------------------
CLASSES = ["Class A", "Class B", "Class C", "Class D"]

DESCRIPCION_CLASES = {
    "Class A": {
        "titulo": "🟢 Clase A: Inmaduro / Totalmente Verde",
        "estado": "Fruto en fase inicial de cosecha o almacenamiento en frío.",
        "consumo": "No apto para consumo fresco inmediato. Alta concentración de almidón no digerible.",
        "aplicacion": "Ideal para transporte de larga distancia o cocción tradicional (plátano verde)."
    },
    "Class B": {
        "titulo": "🟡🟢 Clase B: Inicio de Maduración / Verde-Amarillo",
        "estado": "Inicio de la conversión enzimática de almidones a azúcares simples.",
        "consumo": "Textura firme y sabor ligeramente astringente.",
        "aplicacion": "Punto de llegada a centros de distribución y supermercados."
    },
    "Class C": {
        "titulo": "🟡 Clase C: Maduro Óptimo / Amarillo Brillante",
        "estado": "Punto máximo de calidad organoléptica y balance de azúcares.",
        "consumo": "Óptimo para consumo directo fresco. Máxima digestibilidad.",
        "aplicacion": "Venta directa al consumidor minorista."
    },
    "Class D": {
        "titulo": "🟤 Clase D: Sobre-maduro / Manchas Pardas o Negras",
        "estado": "Fase avanzada de maduración con azúcares concentrados y piel delgada.",
        "consumo": "Sabor muy dulce y textura suave.",
        "aplicacion": "Excelente para repostería, pan de banano, batidos o procesamiento industrial."
    }
}

# Preprocesamiento estándar ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def preprocesar(img: Image.Image) -> torch.Tensor:
    if img.mode != "RGB":
        img = img.convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    return transform(img).unsqueeze(0)

# -------------------------------------------------------------
# 3. Gestor de Inferencia (ExecuTorch INT8)
# -------------------------------------------------------------
class BananaInferenceEngine:
    def __init__(self, pte_path: str):
        if not os.path.exists(pte_path):
            raise FileNotFoundError(f"[ERROR] No se encontró el modelo cuantizado: {pte_path}")
        print(f"[*] Cargando modelo cuantizado ExecuTorch: {pte_path}")
        self.runtime = Runtime.get()
        self.program = self.runtime.load_program(pte_path)
        self.method = self.program.load_method("forward")
        print("[+] ExecuTorch inicializado correctamente.")

    def predict(self, tensor_img: torch.Tensor):
        outputs = self.method.execute([tensor_img])
        logits = outputs[0]
        if isinstance(logits, list):
            logits = logits[0]
        if not isinstance(logits, torch.Tensor):
            logits = torch.from_numpy(np.array(logits))
        probs = torch.softmax(logits[0], dim=0)
        return probs, "ExecuTorch INT8 (XNNPACK)"

# Inicializar motor global con el modelo cuantizado INT8
engine = BananaInferenceEngine(PATH_PTE)

# -------------------------------------------------------------
# 4. Función de Predicción para la Interfaz (Decorada con @spaces.GPU)
# -------------------------------------------------------------
@spaces.GPU
def clasificar_madurez(imagen_pil: Image.Image):
    if imagen_pil is None:
        return None, "", "Por favor carga una imagen de un banano."

    try:
        tensor = preprocesar(imagen_pil)
        probs, backend_name = engine.predict(tensor)

        # Diccionario de probabilidades para gr.Label
        prob_dict = {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))}
        
        # Clase predicha con mayor probabilidad
        top_idx = int(torch.argmax(probs).item())
        clase_top = CLASSES[top_idx]
        info = DESCRIPCION_CLASES[clase_top]

        detalle_md = f\"\"\"
### {info['titulo']}
* **Nivel de Confianza:** {float(probs[top_idx])*100:.2f}%
* **Estado Fisiológico:** {info['estado']}
* **Recomendación de Consumo:** {info['consumo']}
* **Destino Comercial Sugerido:** {info['aplicacion']}
\"\"\"
        info_backend = f"**Motor de Inferencia:** `{backend_name}`"
        return prob_dict, detalle_md, info_backend

    except Exception as e:
        return None, f"Error en inferencia: {str(e)}", "Fallo"

# -------------------------------------------------------------
# 5. Interfaz Gráfica con Gradio Blocks
# -------------------------------------------------------------
with gr.Blocks(title="Detector de Madurez de Bananos - ExecuTorch") as demo:
    gr.Markdown(\"\"\"
    # 🍌 Clasificador de Madurez de Bananos en el Borde (Edge AI)
    ### **Desarrollado con PyTorch, ExecuTorch (INT8) y Algoritmo CREDA**
    *Taller IEEE - Universidad Estatal Península de Santa Elena (UPSE)*
    *Docentes: Luis Chuquimarca (lchuquimarca@upse.edu.ec) & Lucas Iturriago (liturriago@unal.edu.co)*
    \"\"\")
    
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="pil", label="Cargar Imagen de Banano")
            btn_predict = gr.Button("🔍 Analizar Estado de Madurez", variant="primary")
            backend_badge = gr.Markdown("**Motor de Inferencia:** `Detectando...`")

        with gr.Column(scale=1):
            output_label = gr.Label(num_top_classes=4, label="Probabilidades por Clase")
            output_details = gr.Markdown("### Selecciona una imagen y haz clic en 'Analizar'.")

    btn_predict.click(
        fn=clasificar_madurez,
        inputs=[input_image],
        outputs=[output_label, output_details, backend_badge],
        api_name="predict"
    )

if __name__ == "__main__":
    demo.launch(theme=gr.themes.Soft(primary_hue="amber", neutral_hue="slate"))
"""

with open(SPACE_DIR / "app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print(f"[+] Archivo app.py creado en {SPACE_DIR}/app.py")

## 6.4. Generación de Dependencias (`requirements.txt`)

Para Hugging Face Spaces con el SDK nativo de Gradio, definimos las dependencias en `requirements.txt`.
Esto permite que Hugging Face instale las librerías automáticamente y soporte aceleración de hardware (incluyendo ZeroGPU y CPU estándar).

In [ ]:
requirements_content = """torch>=2.1.0
torchvision
numpy
pillow
gradio
executorch
"""

with open(SPACE_DIR / "requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

# Eliminar Dockerfile previo si existiera para evitar conflictos con el SDK de Gradio
if (SPACE_DIR / "Dockerfile").exists():
    (SPACE_DIR / "Dockerfile").unlink()

print(f"[+] Archivo requirements.txt creado en {SPACE_DIR}/requirements.txt")

## 6.5. Generación de los Metadatos `README.md` (Gradio SDK)

Hugging Face requiere un bloque inicial en formato YAML. Especificamos `sdk: gradio` con la versión detectada de Gradio para garantizar compatibilidad con el entorno de ejecución (y evitar el error de ZeroGPU exclusivo de Gradio).

In [ ]:
import gradio as gr

readme_code = f"""---
title: Banana Ripeness Detector - MobileNetV3 ExecuTorch INT8
emoji: 🍌
colorFrom: yellow
colorTo: green
sdk: gradio
sdk_version: {gr.__version__}
app_file: app.py
pinned: false
license: mit
---

# Clasificador de Madurez de Bananos (Edge AI)
Aplicación interactiva desarrollada para el **Taller IEEE** en la **Universidad Estatal Península de Santa Elena (UPSE)**.
* **Arquitectura:** MobileNetV3 Large
* **Dominio:** Class-Regularized Entropy Domain Adaptation (CREDA)
* **Aceleración Edge:** Cuantización INT8 con ExecuTorch y backend XNNPACK
* **Docentes:** Luis Chuquimarca (lchuquimarca@upse.edu.ec) & Lucas Iturriago (liturriago@unal.edu.co)
"""

with open(SPACE_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_code)

print(f"[+] Archivo README.md creado en {SPACE_DIR}/README.md (sdk: gradio)")

## 6.6. Copiado del Modelo Cuantizado al Directorio del Space

Para un despliegue Edge AI genuino, únicamente subimos el artefacto binario optimizado de ExecuTorch (`model_quantized_xnnpack.pte`). No se requiere subir los pesos originales de PyTorch (`model_final.pth`), reduciendo el tamaño a tan solo ~4.3 MB (un 73% más ligero).

In [ ]:
import shutil

source_pte = Path("outputs/mobilenetv3/experiment_3/model_quantized_xnnpack.pte")

# Si existiera una copia previa del modelo .pth en la carpeta local del Space, la eliminamos
if (SPACE_DIR / "model_final.pth").exists():
    (SPACE_DIR / "model_final.pth").unlink()

# Copiar modelo ExecuTorch INT8
if source_pte.exists():
    shutil.copy(source_pte, SPACE_DIR / "model_quantized_xnnpack.pte")
    print(f"[+] Modelo ExecuTorch copiado ({source_pte.stat().st_size / (1024*1024):.2f} MB)")
else:
    print("[!] AVISO: No se encontró model_quantized_xnnpack.pte. Asegúrate de ejecutar la Sección 5.")

print(f"\nContenido del directorio de despliegue '{SPACE_NAME}':")
for file in SPACE_DIR.iterdir():
    print(f"  📄 {file.name} ({file.stat().st_size / 1024:.1f} KB)")

## 6.7. Publicación y Despliegue en Hugging Face

Subiremos la carpeta directamente a Hugging Face usando `HfApi.upload_folder`. Esto evita configuraciones complejas de Git LFS y automatiza el inicio de la compilación en el servidor remoto.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = f"{HF_USER}/{SPACE_NAME}"

# 1. Crear el repositorio en Hugging Face si no existe (usando space_sdk="gradio")
try:
    api.create_repo(repo_id=repo_id, repo_type="space", space_sdk="gradio", exist_ok=True)
    print(f"[+] Espacio '{repo_id}' confirmado en Hugging Face (SDK: gradio).")
except Exception as e:
    print(f"[!] Error o verificación de repositorio: {e}")

# 2. Subir todos los archivos generados
print("[*] Subiendo archivos y modelos al Space...")
api.upload_folder(
    folder_path=str(SPACE_DIR),
    repo_id=repo_id,
    repo_type="space",
    commit_message="Despliegue de MobileNetV3 INT8 con Gradio SDK - Taller IEEE"
)

print(f"\n🚀 [DESPLIEGUE INICIADO EXITOSAMENTE]")
print(f"Visita tu aplicación en vivo: https://huggingface.co/spaces/{repo_id}")

---
# 7. Consumo Remoto de la API del Space con `gradio_client`

Una vez que el Space haya completado la compilación de su contenedor Docker (suele tomar 2 a 3 minutos en el primer despliegue), podemos consumir el endpoint como una API REST desde Python.

In [ ]:
from gradio_client import Client
try:
    from gradio_client import handle_file
except ImportError:
    handle_file = None
import glob

# Seleccionar una imagen de prueba real del dataset
sample_images = glob.glob(f"{DATASET_DIR}/Medium_Variation/test/Class A/*.png")
if sample_images:
    sample_img_path = sample_images[0]
    print(f"[*] Imagen de prueba seleccionada: {sample_img_path}")

    try:
        # Conectar con el cliente de Gradio
        client = Client(f"{HF_USER}/{SPACE_NAME}")
        
        # Preparar payload compatible con Gradio 4 y 5
        img_payload = handle_file(sample_img_path) if handle_file is not None else sample_img_path

        # Enviar petición de inferencia al endpoint registrado /predict
        resultado = client.predict(
            img_payload,
            api_name="/predict"
        )
        print("\n[+] Predicción recibida exitosamente del Space:")
        print(f"  * Probabilidades: {resultado[0]}")
        print(f"  * Detalle y Recomendación:\n{resultado[1]}")
        print(f"  * Backend: {resultado[2]}")
    except Exception as e:
        print(f"[!] Detalle: {e}")
        print(f"Puedes verificar el log de ejecución de tu Space en: https://huggingface.co/spaces/{HF_USER}/{SPACE_NAME}")
else:
    print("[!] No se encontraron imágenes locales en el test set para probar el cliente.")

---
### **Conclusiones del Taller**
1. **Robustez ante Domain Shift:** El algoritmo CREDA permite adaptar modelos entrenados con datos controlados o sintéticos a entornos con variaciones complejas de iluminación sin requerir nuevas anotaciones manuales.
2. **Eficiencia en el Borde con ExecuTorch:** La cuantización estática a INT8 reduce el tamaño de almacenamiento en más de un 70% y reduce la latencia por inferencia en CPUs de bajo costo gracias a las optimizaciones del backend XNNPACK.
3. **Despliegue MLOps Resiliente:** La integración de Docker, Gradio y arquitecturas de contingencia (fallback) garantiza que las soluciones puedan ser prototipadas, auditadas y consumidas de manera fiable en producción.